# GIK-IceChain v2.0 — E2E Pipeline Test (Colab)

Full **C1 → C2 → C3** pipeline on Google Colab, self-contained:
every prerequisite is downloaded from public sources and the IceChunk /
exceedance stores are written to the local Colab filesystem by default.

| Phase | What it does |
|-------|-------------|
| **1** | System setup (uv, eccodes) |
| **2** | Clone repo + install dependencies |
| **3** | Secrets & storage mode (LOCAL by default, MinIO if secrets set) |
| **4** | Download all prerequisites (boundaries, thresholds, ENSO/IOD, CHIRPS rainfall) |
| **5** | Run `gik-icechain run-all` (C1 → C2 → C3) |
| **6** | Verify outputs + interpret results (map, 2yr/5yr risk, top units) |

> **Runtime**: standard CPU. No GPU needed. ~10-20 min total.
>
> **Optional Colab secrets** (🔑 panel, all optional — without them the run is fully local):
> `AWS_ENDPOINT_URL`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` → write stores to your MinIO/S3;
> `HF_TOKEN` → higher HuggingFace rate limits;
> `EARTHDATA_USER`, `EARTHDATA_PASSWORD` → NASA GPM IMERG instead of CHIRPS.

---
## Phase 1 — System Setup

In [ ]:
import sys, shutil, subprocess

py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
assert sys.version_info >= (3, 12), f"Python >= 3.12 required, got {py_ver}"
print(f"OK  Python {py_ver}")

if not shutil.which("uv"):
    subprocess.check_call(["pip", "install", "-q", "uv"])
print("OK  uv")

print("INFO Installing eccodes system library ...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
r = subprocess.run(["apt-get", "install", "-y", "-qq", "libeccodes0", "libeccodes-tools"],
                   capture_output=True)
if r.returncode != 0:  # newer Ubuntu renames the runtime package
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "libeccodes0t64", "libeccodes-tools"],
                          stdout=subprocess.DEVNULL)
print("OK  eccodes")

---
## Phase 2 — Clone Repository + Install Dependencies

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/gik-icechain.git"
BRANCH   = "develop"
REPO_DIR = Path("/content/gik-icechain")

if not (REPO_DIR / "pyproject.toml").exists():
    print(f"INFO Cloning (branch: {BRANCH}) ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH,
                           REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
print(f"OK  Working directory: {Path.cwd()}")

In [ ]:
print("INFO uv sync (installs all dependencies) ...")
subprocess.check_call(["uv", "sync"], stdout=subprocess.DEVNULL)
r = subprocess.run(["uv", "run", "gik-icechain", "--help"], capture_output=True)
print("OK  CLI gik-icechain accessible" if r.returncode == 0 else "FAIL CLI not working")

---
## Phase 3 — Secrets & Storage Mode

Reads optional Colab secrets. With AWS secrets the pipeline writes to your
MinIO/S3; without them it writes IceChunk + Zarr stores to the local
filesystem — no external infrastructure at all.

In [ ]:
def get_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, default)


AWS_ENDPOINT = get_secret("AWS_ENDPOINT_URL")
AWS_KEY      = get_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET   = get_secret("AWS_SECRET_ACCESS_KEY")
HF_TOKEN     = get_secret("HF_TOKEN")
EARTHDATA_U  = get_secret("EARTHDATA_USER")
EARTHDATA_P  = get_secret("EARTHDATA_PASSWORD")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if EARTHDATA_U:
    os.environ["EARTHDATA_USER"] = EARTHDATA_U
    os.environ["EARTHDATA_PASSWORD"] = EARTHDATA_P

LIVE = bool(AWS_ENDPOINT and AWS_KEY and AWS_SECRET)
OUTPUT = Path("results/e2e_colab")

if LIVE:
    os.environ["AWS_ENDPOINT_URL"]      = AWS_ENDPOINT
    os.environ["AWS_ACCESS_KEY_ID"]     = AWS_KEY
    os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET
    os.environ["AWS_REGION"]            = "us-east-1"
    CONFIG = "configs/default.yaml"
    print(f"OK  LIVE mode — stores on {AWS_ENDPOINT}")
else:
    # Thin override deep-merged on top of configs/default.yaml: local stores.
    local_cfg = f"""outputs:
  icechunk_store_uri: "{REPO_DIR / OUTPUT / 'icechunk-store'}"
  icechunk_store_region: "eu-west-1"
  endpoint_url: ""
  exceedance_store_uri: "{REPO_DIR / OUTPUT / 'exceedance-zarr'}"
  exceedance_icechunk_uri: ""
  risk_icechunk_uri: ""
  risk_output_dir: "{OUTPUT}/admin1_risk/"
"""
    Path("configs/colab_e2e.yaml").write_text(local_cfg)
    CONFIG = "configs/colab_e2e.yaml"
    print("OK  LOCAL mode — stores on the Colab filesystem (no secrets needed)")
print(f"    config: {CONFIG}")

---
## Phase 4 — Download All Prerequisites

Public sources only: geoBoundaries (admin-1), HuggingFace (CMORPH return
periods → GEV threshold files), NOAA (ENSO/IOD index), CHC (CHIRPS daily
rainfall — GPM-compatible, no account needed).

In [ ]:
from datetime import date, timedelta

# ECMWF open-data S3 has rolling retention -> use recent dates.
# Pin START_DATE/END_DATE manually to test a specific historical window.
END_DATE   = (date.today() - timedelta(days=2)).isoformat()
START_DATE = (date.today() - timedelta(days=4)).isoformat()
print(f"Pipeline window: {START_DATE} -> {END_DATE}")

In [ ]:
tools = ["uv", "run", "python3", "scripts/tools.py"]

print("INFO admin boundaries + CMORPH return periods + ENSO/IOD ...")
subprocess.check_call([*tools, "download", "--component", "all"])

print("\nINFO GEV threshold files ...")
subprocess.check_call([*tools, "download-thresholds"])

gpm_source = "nasa" if EARTHDATA_U else "chirps"
print(f"\nINFO daily rainfall ({gpm_source}) for the C3 Obs_Antecedent node ...")
subprocess.check_call([*tools, "download-gpm", "--source", gpm_source,
                       "--start", START_DATE, "--end", END_DATE])
print("\nOK  all prerequisites downloaded")

---
## Phase 5 — E2E Pipeline (C1 → C2 → C3)

In [ ]:
import shutil as _sh

if OUTPUT.exists():
    _sh.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)

env = {**os.environ, "ECCODES_PYTHON_USE_FINDLIBS": "1"}
cmd = ["uv", "run", "gik-icechain", "run-all",
       "--start", START_DATE, "--end", END_DATE,
       "--config", CONFIG, "--output", str(OUTPUT)]
print("INFO", " ".join(cmd), "\n")
rc = subprocess.run(cmd, env=env).returncode
print(f"\n{'OK  pipeline finished' if rc == 0 else f'FAIL exit code {rc}'}")

---
## Phase 6 — Verification & Interpretation

In [ ]:
import glob, json

risk_dir = OUTPUT / "admin1_risk"
files = sorted(glob.glob(str(risk_dir / "*_risk_scores.json")))
assert files, f"No risk score files in {risk_dir} — check the run-all log above."

print(f"OK  {len(files)} day(s) of risk scores in {risk_dir}\n")
print(f"  {'Date':<12} {'Units':>6}  Risk distribution (primary RP)")
for f in files:
    data = json.load(open(f))
    units = data["units"]
    counts: dict[str, int] = {}
    for u in units.values():
        counts[u["risk_label"]] = counts.get(u["risk_label"], 0) + 1
    dist = ", ".join(f"{k}: {v}" for k, v in sorted(counts.items()))
    print(f"  {data['date']:<12} {len(units):>6}  {dist}")

In [ ]:
# 2yr vs 5yr view (risk_by_rp) for the last day
last = json.load(open(files[-1]))
units = last["units"]

by_rp: dict[str, dict[str, int]] = {}
for u in units.values():
    for rp, r in u.get("risk_by_rp", {}).items():
        by_rp.setdefault(rp, {})
        by_rp[rp][r["risk_label"]] = by_rp[rp].get(r["risk_label"], 0) + 1

print(f"Return-period comparison — {last['date']}")
for rp in sorted(by_rp, key=int):
    dist = ", ".join(f"{k}: {v}" for k, v in sorted(by_rp[rp].items()))
    print(f"  {rp}yr : {dist}")
print()

top = sorted(units.items(), key=lambda kv: kv[1].get("p_red", 0), reverse=True)[:10]
print(f"Top 10 admin-1 units by P(Red) — {last['date']}")
print(f"  {'pcode':<26} {'label':<8} {'p_red':>6} {'p_orange':>9} {'exc 24h':>8} {'API mm':>7}")
for pcode, u in top:
    print(f"  {pcode:<26} {u['risk_label']:<8} {u['p_red']:>6.3f} "
          f"{u['p_orange']:>9.3f} {u.get('exceedance_24h', 0):>8.3f} {u.get('api_mm', 0):>7.1f}")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

COLORS = {"Green": "#2ecc71", "Yellow": "#f1c40f", "Orange": "#e67e22",
          "Red": "#e74c3c", "No_Data": "#cccccc"}

admin = gpd.read_file("data/admin_boundaries/east_africa_admin1.geojson")
admin["risk_label"] = admin["admin1_pcode"].map(
    lambda p: units.get(p, {}).get("risk_label", "No_Data"))
admin["color"] = admin["risk_label"].map(COLORS)

fig, ax = plt.subplots(figsize=(9, 9))
admin.plot(ax=ax, color=admin["color"], edgecolor="white", linewidth=0.3)
ax.set_title(f"East Africa flood risk — {last['date']} (5yr return period)")
ax.set_axis_off()
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORS.values()]
ax.legend(handles, COLORS.keys(), loc="lower left")
plt.show()

### How to read these results

- **Risk labels** come from the CRMA Bayesian Network: `P(Green..Red)` sum to 1
  per unit; the label is the argmax. `No_Data` means the unit had no finite
  exceedance aggregate (coverage below `min_coverage_fraction`).
- **`risk_by_rp`** gives the same inference per return period. The 2yr view uses
  its own hazard calibration (`hazard_thresholds_by_rp`) and its own dynamic BN
  state; read it as a sensitivity view — the CRMA CPTs were elicited against
  5yr exceedance semantics (see `RESULT.md`).
- **`exceedance_24h`** is the admin-aggregated probability that 24h rainfall
  exceeds the GEV threshold for the primary RP; **`API mm`** is the antecedent
  precipitation index driving the soil-memory nodes.
- A short window right after `START_DATE` keeps `consecutive_signal_days` and
  API near their initial values — multi-day persistence effects only appear in
  longer runs.

In [ ]:
print("=" * 44)
print(" Summary")
print("=" * 44)
print(f"  Mode       : {'LIVE (' + AWS_ENDPOINT + ')' if LIVE else 'LOCAL filesystem'}")
print(f"  Date range : {START_DATE} -> {END_DATE}")
print(f"  Config     : {CONFIG}")
print(f"  Output     : {OUTPUT}")
print(f"  Days       : {len(files)}  |  Units/day: {len(units)}")
print()
print("All phases passed.")